In [1]:
import pandas as pd
import numpy as np
import pickle
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix)
from IPython.display import display, Markdown

X_train = pd.read_pickle('../data/processed/tree_ready/X_train.pkl')
X_test = pd.read_pickle('../data/processed/tree_ready/X_test.pkl')
y_train = pd.read_pickle('../data/processed/tree_ready/y_train.pkl')
y_test = pd.read_pickle('../data/processed/tree_ready/y_test.pkl')
from sklearn.model_selection import train_test_split

# Carve a validation split out of TRAINING data for early stopping.
# Never use X_test/y_test here — that leaks test data into model selection.
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.15, stratify=y_train, random_state=42
)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results = []
interpretation_log = []

def evaluate_and_interpret(name, model, X_test, y_test, best_params=None, cv_score=None):
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else None

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba) if y_proba is not None else np.nan
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()

    metrics = {'model': name, 'accuracy': acc, 'precision': prec,
               'recall': rec, 'f1': f1, 'auc': auc}
    results.append(metrics)

    recall_quality = "strong" if rec > 0.7 else "moderate" if rec > 0.5 else "weak"
    auc_quality = "strong" if auc > 0.8 else "moderate" if auc > 0.65 else "weak"

    md = f"""### {name}

**Best Hyperparameters (5-fold Stratified CV, optimized for recall):** {best_params if best_params else 'N/A'}
**CV Recall Score:** {f'{cv_score:.3f}' if cv_score else 'N/A'}

**Test Set Metrics:**
- Accuracy: {acc:.3f}
- Precision: {prec:.3f}
- Recall (Sensitivity): {rec:.3f}
- F1 Score: {f1:.3f}
- AUC: {auc:.3f}

**Confusion Matrix:** TN={tn}, FP={fp}, FN={fn}, TP={tp}

**Interpretation:**
- Correctly identifies {rec*100:.1f}% of truly anemic women (recall) — {recall_quality} for this health screening context.
- Of women predicted anemic, {prec*100:.1f}% actually are (precision).
- AUC of {auc:.3f} indicates {auc_quality} discriminative ability.
- False negatives (missed anemia cases): {fn}.
"""
    display(Markdown(md))
    interpretation_log.append(md)
    return metrics

In [2]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

rf_param_grid = {
    'n_estimators': [200, 300, 400],
    'max_depth': [8, 10, 12],
    'min_samples_leaf': [1, 3, 5]
}

rf_grid = GridSearchCV(
    RandomForestClassifier(
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    ),
    rf_param_grid,
    cv=skf,
    scoring='recall',
    n_jobs=-1
)

rf_grid.fit(X_train, y_train)

best_rf = rf_grid.best_estimator_

evaluate_and_interpret(
    'Random Forest',
    best_rf,
    X_test,
    y_test,
    best_params=rf_grid.best_params_,
    cv_score=rf_grid.best_score_
)

### Random Forest

**Best Hyperparameters (5-fold Stratified CV, optimized for recall):** {'max_depth': 8, 'min_samples_leaf': 5, 'n_estimators': 200}
**CV Recall Score:** 0.555

**Test Set Metrics:**
- Accuracy: 0.710
- Precision: 0.541
- Recall (Sensitivity): 0.531
- F1 Score: 0.536
- AUC: 0.706

**Confusion Matrix:** TN=366, FP=96, FN=100, TP=113

**Interpretation:**
- Correctly identifies 53.1% of truly anemic women (recall) — moderate for this health screening context.
- Of women predicted anemic, 54.1% actually are (precision).
- AUC of 0.706 indicates moderate discriminative ability.
- False negatives (missed anemia cases): 100.


{'model': 'Random Forest',
 'accuracy': 0.7096296296296296,
 'precision': 0.5406698564593302,
 'recall': 0.5305164319248826,
 'f1': 0.5355450236966824,
 'auc': 0.7058817551775298}

In [3]:
lgbm_param_grid = {
    'n_estimators': [200, 300, 400],
    'max_depth': [4, 6, 8],
    'learning_rate': [0.03, 0.05, 0.1]
}

lgbm_grid = GridSearchCV(
    LGBMClassifier(class_weight='balanced', random_state=42),
    lgbm_param_grid, cv=skf, scoring='recall', n_jobs=-1
)
lgbm_grid.fit(X_train, y_train)
best_lgbm = lgbm_grid.best_estimator_

evaluate_and_interpret('LightGBM', best_lgbm, X_test, y_test,
                        best_params=lgbm_grid.best_params_, cv_score=lgbm_grid.best_score_)

[LightGBM] [Info] Number of positive: 851, number of negative: 1845
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001233 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 602
[LightGBM] [Info] Number of data points in the train set: 2696, number of used features: 19
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf

### LightGBM

**Best Hyperparameters (5-fold Stratified CV, optimized for recall):** {'learning_rate': 0.05, 'max_depth': 4, 'n_estimators': 200}
**CV Recall Score:** 0.564

**Test Set Metrics:**
- Accuracy: 0.689
- Precision: 0.506
- Recall (Sensitivity): 0.559
- F1 Score: 0.531
- AUC: 0.700

**Confusion Matrix:** TN=346, FP=116, FN=94, TP=119

**Interpretation:**
- Correctly identifies 55.9% of truly anemic women (recall) — moderate for this health screening context.
- Of women predicted anemic, 50.6% actually are (precision).
- AUC of 0.700 indicates moderate discriminative ability.
- False negatives (missed anemia cases): 94.


{'model': 'LightGBM',
 'accuracy': 0.6888888888888889,
 'precision': 0.5063829787234042,
 'recall': 0.5586854460093896,
 'f1': 0.53125,
 'auc': 0.6999878056216084}

In [4]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

# Step 1: GridSearchCV to find best max_depth / learning_rate (structural hyperparameters)
xgb_param_grid = {
    'max_depth': [4, 6, 8],
    'learning_rate': [0.03, 0.05, 0.1]
}

xgb_grid = GridSearchCV(
    XGBClassifier(n_estimators=400, scale_pos_weight=scale_pos_weight,
                  random_state=42, eval_metric='logloss'),
    xgb_param_grid, cv=skf, scoring='recall', n_jobs=-1
)
xgb_grid.fit(X_train, y_train)

# Step 2: refit final model using best structural params + early stopping
# to find the right number of boosting rounds automatically (avoids overfitting seen in learning curve)
best_params = xgb_grid.best_params_

best_xgb = XGBClassifier(
    max_depth=best_params['max_depth'],
    learning_rate=best_params['learning_rate'],
    n_estimators=400,
    scale_pos_weight=scale_pos_weight,
    eval_metric='logloss',
    early_stopping_rounds=20,
    random_state=42
)
best_xgb.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)  # X_val, not X_test

print("Best structural params (from CV):", best_params)
print("Best iteration (early stopping):", best_xgb.best_iteration)

evaluate_and_interpret('XGBoost', best_xgb, X_test, y_test,
                        best_params={**best_params, 'best_iteration': best_xgb.best_iteration},
                        cv_score=xgb_grid.best_score_)

Best structural params (from CV): {'learning_rate': 0.05, 'max_depth': 4}
Best iteration (early stopping): 91


### XGBoost

**Best Hyperparameters (5-fold Stratified CV, optimized for recall):** {'learning_rate': 0.05, 'max_depth': 4, 'best_iteration': 91}
**CV Recall Score:** 0.561

**Test Set Metrics:**
- Accuracy: 0.716
- Precision: 0.549
- Recall (Sensitivity): 0.549
- F1 Score: 0.549
- AUC: 0.705

**Confusion Matrix:** TN=366, FP=96, FN=96, TP=117

**Interpretation:**
- Correctly identifies 54.9% of truly anemic women (recall) — moderate for this health screening context.
- Of women predicted anemic, 54.9% actually are (precision).
- AUC of 0.705 indicates moderate discriminative ability.
- False negatives (missed anemia cases): 96.


{'model': 'XGBoost',
 'accuracy': 0.7155555555555555,
 'precision': 0.5492957746478874,
 'recall': 0.5492957746478874,
 'f1': 0.5492957746478874,
 'auc': 0.7047436131943176}

In [5]:
output_path = '../data/processed/ensemble_ml_interpretation.txt'
with open(output_path, 'w', encoding='utf-8') as f:
    f.write('\n\n---\n\n'.join(interpretation_log))
print(f"Saved interpretations to {output_path}")

results_df = pd.DataFrame(results)
results_df.to_csv('../results/metrics/ensemble_ml_metrics.csv', index=False)
print(results_df)


Saved interpretations to ../data/processed/ensemble_ml_interpretation.txt
           model  accuracy  precision    recall        f1       auc
0  Random Forest  0.709630   0.540670  0.530516  0.535545  0.705882
1       LightGBM  0.688889   0.506383  0.558685  0.531250  0.699988
2        XGBoost  0.715556   0.549296  0.549296  0.549296  0.704744


In [6]:
with open('../models/ml/random_forest.pkl', 'wb') as f:
    pickle.dump(best_rf, f)
with open('../models/ml/xgboost.pkl', 'wb') as f:
    pickle.dump(best_xgb, f)
with open('../models/ml/lightgbm.pkl', 'wb') as f:
    pickle.dump(best_lgbm, f)
with open('../models/ml/xgboost_early_stopped_diag.pkl', 'wb') as f:
    pickle.dump(best_xgb, f)